In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import pandas as pd
import numpy as np
import random
import torch
from google.colab import drive
from sklearn.metrics import accuracy_score, f1_score, recall_score

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def make_prompt(text):
  prompt = """Task: Identify whether the given text is generated by an Language model or is human-written. Choose your answer from the given answer choices.
  Answer choices:
      LM
      Human
      Unsure
  Text: """ + text + """
  Answer: """
  return prompt

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    print(predictions, "\n", labels)
    predictions = predictions.argmax(axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    recall = recall_score(labels, predictions)
    return {"f1": f1, 'accuracy': accuracy, 'recall': recall}

In [ ]:
def train_model(model, tokenizer, train_dataset, val_dataset):
    config = LoraConfig(
        r=16,
        lora_alpha=8,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, config)

    training_args = TrainingArguments(
        output_dir="/content/drive/MyDrive/Training",
        learning_rate=3e-5,
        num_train_epochs=1,
        logging_steps=25,
        save_steps=25,
        save_strategy="steps",
        report_to="tensorboard",
        remove_unused_columns=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        fp16=True,
        optim="adafactor"
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt")

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator
    )

    model.config.use_cache = False

    print('Дообучение')
    trainer.train("/content/drive/MyDrive/Training/checkpoint-2600")

    trainer.save_model("/content/drive/MyDrive/Training/model")
    tokenizer.save_pretrained("/content/drive/MyDrive/Training/model")

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
      "Qwen/Qwen3-8B",
      device_map="cuda:0",
      quantization_config=quantization_config,
      trust_remote_code=True
  )
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B", trust_remote_code=True)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [ ]:
def map_for_training(raw):
    system_prompt = 'You are an expert in text classification'
    user_prompt = f"""Identify whether the given text is generated by an Language model or is human-written. Write your answer in one word, choose it from the given answer choices.
Answer choices:
    LM
    Human
    Unsure
Text: {raw["text"]}
Answer:"""
    columns = [
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt},
          {"role": "assistant", "content": raw["label"]}
    ]

    full_text = tokenizer.apply_chat_template(columns, tokenize=False)

    tokenized = tokenizer(
        full_text,
        truncation=True,
        max_length=512,
        padding=False,
        add_special_tokens=False
    )

    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized

In [ ]:
def map_for_training_class(raw):

    system_prompt = 'You are an expert in text classification'

    user_prompt = f"""Identify the language model the given text is generated with. Write your answer in one word, choose it from the given answer choices.
Answer choices:
    GPT-5
    GPT-4
    GPT-4 Omni
    DeepSeek-V3
    PaLM-2
    GPT-3.5
    GPT-4 Turbo
    Falcon-180B
    Claude 3 Opus
    Gemini-Pro
    GLM
    Goliath
    Command R
    Llama3
Text: {raw["text"]}
Answer:"""
    columns = [
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt},
          {"role": "assistant", "content": raw["model"]}
    ]

    full_text = tokenizer.apply_chat_template(columns, tokenize=False)

    tokenized = tokenizer(
        full_text,
        truncation=True,
        max_length=512,
        padding=False,
        add_special_tokens=False
    )

    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized

In [ ]:
modelclass_answers = ["GPT-5", "GPT-4", "GPT-4 Omni", "DeepSeek-V3", "PaLM-2",
                      "GPT-3.5", "GPT-4 Turbo", "Falcon-180B", "Claude 3 Opus",
                      "Gemini-Pro", "GLM", "Goliath", "Command R", "Llama3"]
modelnames_conv = {'Claude 3 Opus' : "Claude 3 Opus",
                   'Command R+' : "Command R",
                   'DeepSeek-V3.2' : "DeepSeek-V3",
                   'Falcon-180B' : "Falcon-180B",
                   'GLM-130B' : "GLM",
                   'GPT-3.5' : "GPT-3.5",
                   'GPT-4' : "GPT-4",
                   'GPT-5' : "GPT-5",
                   'GPT4 Omni' : "GPT-4 Omni",
                   'GPT4 Turbo' : "GPT-4 Turbo",
                   'Gemini-Pro' : "Gemini-Pro",
                   'Goliath-120B' : "Goliath",
                   'Llama3 70B' : "Llama3",
                   'PaLM-2' : "PaLM-2"}

In [ ]:
import datasets
seed_everything(42)
# train = pd.read_csv("/content/drive/MyDrive/Datasets/train30000.csv")
# dataset_train = Dataset.from_pandas(train)
# modelclass_train = train.loc[(train['model'] != 'human')]
# modelclass_train['model'] = [modelnames_conv[model] for model in modelclass_train['model']]
# dataset_modelclass_train = Dataset.from_pandas(modelclass_train)
# processed_train = dataset_train.map(map_for_training, remove_columns=dataset_train.column_names)
# processed_modelclass_train = dataset_modelclass_train.map(map_for_training_class, remove_columns=dataset_modelclass_train.column_names)

# united_train = datasets.concatenate_datasets([processed_train, processed_modelclass_train])
united_train = datasets.load_from_disk("/content/drive/MyDrive/Datasets/United dataset")

In [ ]:
train_model(model, tokenizer, united_train, None)

Дообучение


Step,Training Loss
2625,1.458424
2650,1.474804
2675,1.501601
2700,1.429428
2725,1.416825
2750,1.432133
2775,1.488791
2800,1.480740
